In [ ]:
import coiled
import xarray as xr
import zarr

from srm import catalog

zarr.config.set({"async.concurrency": 64})

In [ ]:
cluster = coiled.Cluster(
    name="srm-fine-to-coarse",
    n_workers=[20, 40],
    region="us-west-2",
    worker_vm_types="c8g.large",
    scheduler_vm_types=["c8g.xlarge"],
    spot_policy="spot_with_fallback",
    tags={"Project": "SRM"},
)

client = cluster.get_client()
client

## These calcs should all be lazy and only take a few seconds

In [ ]:
# coarse historical GCM grid - Note, single time slice, single var
ds_coarse_grid = catalog.get("CESM2-WACCM-Historical-icechunk").to_xarray()[["tasmax"]].isel(time=0)

# fine ERA5 - single var
ds_fine_grid = catalog.get("ERA5").to_xarray()[["tasmax"]]

# create a target grid from the GCM coarse dataset
target_grid = ds_coarse_grid[["lat", "lon"]].drop_vars("time").reset_coords(drop=True)

# use xarray regird
ds_fine_regridded = ds_fine_grid.regrid.conservative(target_grid, latitude_coord="lat")

## Calling to_zarr or to_icechunk would trigger the computation. 

In [ ]:
%%time
# ~3:30 minutes
# We could speed this up with obstore + zarr backend or icechunk!
# ~ 14.5MB task graph - so not too bad
# ds_fine_regridded.to_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", mode="w")

In [ ]:
# shutdown the coiled cluster
client.shutdown()

In [ ]:
rtds = xr.open_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", chunks="auto")

In [ ]:
rtds.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()

In [ ]:
ds_coarse_grid["tasmax"].sel(lat=slice(25, 50), lon=slice(-100, -70)).plot()

In [ ]:
ds_fine_grid.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()

In [ ]:
# import obstore as obs
# from obstore.store import from_url
# from zarr.storage import ObjectStore
# import boto3
# sesh = boto3.Session()
# creds = sesh.get_credentials()

# store = from_url(url='s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-obstore',
#     aws_access_key_id=creds.access_key,
#     aws_secret_access_key=creds.secret_key, region='us-west-2')
# zstore = ObjectStore(store)

# ds_fine_regridded.to_zarr(zstore, mode="w")
